<a href="https://colab.research.google.com/github/Guliko24/PubMed_Fetch/blob/main/Week1_Day3_Vector_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip -q install sentence-transformers rank-bm25 numpy

In [2]:
from pathlib import Path
import subprocess
import sys

GITHUB_USERNAME = "Guliko24"
REPO_NAME = "PubMed_Fetch"

repo_dir = Path("/content") / REPO_NAME
repo_url = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

if not repo_dir.exists():
    subprocess.run(["git", "clone", repo_url, str(repo_dir)], check=True)
else:
    print(f"Using existing clone: {repo_dir}")

sys.path.insert(0, str(repo_dir))
print("Repository:", repo_dir)

Repository: /content/PubMed_Fetch


In [ ]:
# Only needed when the repo is already cloned and GitHub has newer commits:
# !git -C /content/YOUR_REPOSITORY_NAME pull

In [3]:
import search_utils

print("Imported utility from:", search_utils.__file__)
print("Embedding model:", search_utils.MODEL_NAME)

assert Path(search_utils.__file__).resolve() == (
    repo_dir / "search_utils.py"
).resolve(), "Python imported a different search_utils.py file!"

Imported utility from: /content/PubMed_Fetch/search_utils.py
Embedding model: sentence-transformers/all-MiniLM-L6-v2


In [4]:
DATA_FILE = repo_dir / "pubmed_abstracts.json"

# If your JSON is on Google Drive instead, use:
# from google.colab import drive
# drive.mount("/content/drive")
# DATA_FILE = Path(
#     "/content/drive/MyDrive/ontology-aware-litminer/pubmed_abstracts.json"
# )

print("Looking for data at:", DATA_FILE)
print("File exists:", DATA_FILE.exists())

documents = search_utils.load_pubmed_data(DATA_FILE)
print(f"Loaded {len(documents)} documents.")
print("First PMID:", documents[0]["pmid"])
print("First title:", documents[0]["title"])

Looking for data at: /content/PubMed_Fetch/pubmed_abstracts.json
File exists: True
Loaded 10 documents.
First PMID: 34582785
First title: Single-nucleus transcriptome analysis reveals cell-type-specific molecular signatures across reward circuitry in the human brain.


In [5]:
import hashlib
import json
import numpy as np

texts = [
    f"{doc['title']}. {doc['abstract']}"
    for doc in documents
]

fingerprint_input = json.dumps(
    {
        "model": search_utils.MODEL_NAME,
        "documents": [
            {
                "pmid": doc["pmid"],
                "title": doc["title"],
                "abstract": doc["abstract"],
            }
            for doc in documents
        ],
    },
    sort_keys=True,
).encode("utf-8")

fingerprint = hashlib.sha256(fingerprint_input).hexdigest()[:16]
cache_file = repo_dir / f"day3_embeddings_{fingerprint}.npy"

if cache_file.exists():
    doc_embeddings = np.load(cache_file)
    print("Loaded cached embeddings:", cache_file)
else:
    doc_embeddings = search_utils.generate_embeddings(texts)
    np.save(cache_file, doc_embeddings)
    print("Saved embeddings:", cache_file)

print("Embeddings shape:", doc_embeddings.shape)
print("Documents:", len(documents))

assert doc_embeddings.ndim == 2
assert doc_embeddings.shape[0] == len(documents)
assert np.allclose(
    np.linalg.norm(doc_embeddings, axis=1),
    1.0,
    atol=1e-3,
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Saved embeddings: /content/PubMed_Fetch/day3_embeddings_3071cbfb6ef3dc37.npy
Embeddings shape: (10, 384)
Documents: 10


In [6]:
query = "human brain cell types identified by single-cell transcriptomics"

vector_results = search_utils.search_vectors(
    query=query,
    doc_embeddings=doc_embeddings,
    documents=documents,
    top_k=5,
)

print("VECTOR SEARCH")
print("Query:", query)

for rank, result in enumerate(vector_results, start=1):
    doc = result["doc"]
    print(
        f"{rank}. cosine={result['score']:.3f} | "
        f"PMID={doc['pmid']} | {doc['title']}"
    )

VECTOR SEARCH
Query: human brain cell types identified by single-cell transcriptomics
1. cosine=0.793 | PMID=41053013 | Mapping human brain cell type origin and diseases through single-cell transcriptomics.
2. cosine=0.647 | PMID=34582785 | Single-nucleus transcriptome analysis reveals cell-type-specific molecular signatures across reward circuitry in the human brain.
3. cosine=0.644 | PMID=41456076 | Exploring cellular heterogeneity: single-cell and spatial transcriptomics of Alzheimer's disease brains and iPSC-derived microglia.
4. cosine=0.546 | PMID=40585969 | Single-cell transcriptomics of vascularized human brain organoids decipher lineage-specific stress adaptation in fetal hypoxia-reoxygenation injury.
5. cosine=0.545 | PMID=36544231 | Spatially resolved transcriptomics reveals genes associated with the vulnerability of middle temporal gyrus in Alzheimer's disease.


Compare with Day 2 BM25
This is a useful sanity check, not yet hybrid search

In [7]:
bm25_index = search_utils.build_bm25_index(documents)
bm25_results = search_utils.search_bm25(
    bm25_index=bm25_index,
    documents=documents,
    query=query,
    top_k=5,
)

print("BM25 SEARCH")
print("Query:", query)

for rank, result in enumerate(bm25_results, start=1):
    doc = result["doc"]
    print(
        f"{rank}. BM25={result['score']:.3f} | "
        f"PMID={doc['pmid']} | {doc['title']}"
    )

BM25 SEARCH
Query: human brain cell types identified by single-cell transcriptomics
1. BM25=4.558 | PMID=41053013 | Mapping human brain cell type origin and diseases through single-cell transcriptomics.
2. BM25=3.834 | PMID=34582785 | Single-nucleus transcriptome analysis reveals cell-type-specific molecular signatures across reward circuitry in the human brain.
3. BM25=3.097 | PMID=40701154 | Epigenome profiling identifies H3K27me3 regulation of extracellular matrix composition in human corticogenesis.
4. BM25=2.815 | PMID=40585969 | Single-cell transcriptomics of vascularized human brain organoids decipher lineage-specific stress adaptation in fetal hypoxia-reoxygenation injury.
5. BM25=2.594 | PMID=11847215 | Neutrophil-derived glutamate regulates vascular endothelial barrier function.
